# TSF (Temporal Shell Framework) — End‑to‑End Demo

**What this notebook does:**
1. Automatically **downloads real EEG data** (EEGBCI from PhysioNet) using MNE.
2. Preprocesses and epochs the data.
3. Computes **wavelet transforms** (Morlet) and builds a **cross‑scale empirical coupling** matrix with **zero diagonal** (no self‑interactions).
4. **Fits the TSF kernel parameters** \((c_0, \beta, \gamma, \alpha_{ord})\) and estimates the **scaling factor** \(\lambda\).
5. Saves plots and prints key metrics.

**Notes**
- Default source is EEGBCI (robust + small). You can switch to OpenNeuro in the optional cell.
- This is CPU‑friendly and runs in Colab or locally. No hidden data required.


In [ ]:
# --- Parameters you can tweak ---
DATA_SOURCE = "EEGBCI"   # options: "EEGBCI" (default). See optional OpenNeuro cell below.
SUBJECT = 1               # EEGBCI subject id (1..109)
RUNS = [4, 10, 14]        # Typical motor-imagery runs
RESAMPLE_HZ = 250
HP, LP = 1.0, 45.0        # bandpass
EPOCH_DUR, EPOCH_OVERLAP = 2.0, 1.0
N_SHELLS = 24             # number of wavelet scales / shells
WAVELET_NAME = 'morl'     # Morlet
RANDOM_STATE = 42

print({k:v for k,v in list(globals().items()) if k in ['DATA_SOURCE','SUBJECT','RUNS','RESAMPLE_HZ','HP','LP','EPOCH_DUR','EPOCH_OVERLAP','N_SHELLS','WAVELET_NAME']})

In [ ]:
# Install dependencies (Colab/local). If already installed locally, pip will be quick.
try:
    import mne, pywt
except Exception:
    pass

import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split('==')[0])
    except Exception:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

pip_install('mne')
pip_install('pywavelets')
pip_install('matplotlib')
pip_install('numpy')
pip_install('scipy')

import numpy as np, matplotlib.pyplot as plt
import mne, pywt
from scipy.optimize import minimize
from pathlib import Path
mne.set_log_level('WARNING')
np.random.seed(RANDOM_STATE)

OUT = Path('tsf_outputs'); OUT.mkdir(exist_ok=True)
print("Environment ready.")

## 1) Download real EEG data (EEGBCI) and preprocess
This uses **MNE**'s dataset fetcher to pull from PhysioNet automatically.

In [ ]:
def load_eegbci(subject=1, runs=[4,10,14], resample_hz=250, hp=1.0, lp=45.0):
    """Download EEGBCI runs for a subject and return a preprocessed Raw object."""
    from mne.datasets import eegbci
    files = eegbci.load_data(subject, runs)
    raws = [mne.io.read_raw_edf(f, preload=True, verbose='ERROR') for f in files]
    raw = mne.concatenate_raws(raws)
    # Standard montage & filtering
    try:
        raw.set_montage('standard_1020')
    except Exception:
        pass
    raw.filter(hp, lp, fir_design='firwin')
    raw.resample(resample_hz)
    raw.pick_types(eeg=True, eog=False, ecg=False, stim=False)
    return raw

if DATA_SOURCE == 'EEGBCI':
    raw = load_eegbci(SUBJECT, RUNS, RESAMPLE_HZ, HP, LP)
    print(raw)
else:
    raise ValueError("Unsupported DATA_SOURCE in this cell. See optional OpenNeuro cell below.")

# Create simple fixed-length epochs (agnostic to event codes)
epochs = mne.make_fixed_length_epochs(raw, duration=EPOCH_DUR, overlap=EPOCH_OVERLAP, preload=True)
print(epochs)

## 2) Wavelet transform → shells and empirical cross‑scale coupling
We compute a **continuous wavelet transform (CWT)** per channel, convert to **power** (magnitude²),
then compute a **scale×scale** correlation matrix per epoch and channel, averaging across epochs and channels.
Finally, we **enforce the zero diagonal** (no self-interaction).

In [ ]:
def cwt_power(data_1d, n_scales=N_SHELLS, wavelet=WAVELET_NAME):
    # pywt.cwt expects array and scales; we use geometric scales
    scales = np.geomspace(2, 64, num=n_scales)
    coeffs, _ = pywt.cwt(data_1d, scales, wavelet)
    power = np.abs(coeffs)**2  # (n_scales, n_times)
    return power, scales

def empirical_cross_scale_coupling(epochs, n_scales=N_SHELLS):
    X = epochs.get_data()  # (n_epochs, n_ch, n_times)
    n_epochs, n_ch, n_times = X.shape
    C_accum = np.zeros((n_scales, n_scales), dtype=float)
    count = 0
    for e in range(n_epochs):
        for ch in range(n_ch):
            power, scales = cwt_power(X[e, ch, :], n_scales=n_scales)
            # Correlation across scales (rows) over time
            R = np.corrcoef(power)
            C_accum += np.nan_to_num(R)
            count += 1
    C_emp = C_accum / max(1, count)
    # Zero diagonal (no self-interactions)
    np.fill_diagonal(C_emp, 0.0)
    return C_emp, scales

C_emp, scales = empirical_cross_scale_coupling(epochs, n_scales=N_SHELLS)
print("C_emp shape:", C_emp.shape)

# Plot empirical coupling
plt.figure(figsize=(5,4))
plt.imshow(C_emp, origin='lower', aspect='auto')
plt.title('Empirical cross-scale coupling (zero-diagonal)')
plt.xlabel('Shell j'); plt.ylabel('Shell i')
plt.colorbar()
plt.tight_layout()
plt.savefig(OUT/'empirical_coupling.png', dpi=160)
plt.show()

## 3) Fit TSF kernel (\(c_0, \beta, \gamma, \alpha_{ord}\)) and estimate \(\lambda\)
We fit your parametric kernel to the empirical cross‑scale matrix via constrained least squares.
For the ordinal term we use a computable surrogate \(\mathrm{ord}(n)=\log_2(1+n)\), and we estimate \(\lambda\)
by a log‑linear fit of scale vs shell index.

In [ ]:
def estimate_lambda(scales):
    # Fit log(scales) ~ a + b * i  => lambda = exp(b)
    i = np.arange(len(scales))
    y = np.log(scales)
    A = np.vstack([np.ones_like(i), i]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    lam = float(np.exp(b))
    return lam

def tsf_kernel(params, n_shells):
    c0, beta, gamma, aord = params
    idx = np.arange(n_shells)
    D = np.abs(idx[:,None] - idx[None,:])
    Ord = np.log2(1.0 + D)  # surrogate for ord(n)
    K = c0 / (1.0 + beta * (D**np.maximum(gamma, 1e-9))) * np.exp(-aord * Ord)
    np.fill_diagonal(K, 0.0)
    return K

def fit_tsf(C_emp):
    n = C_emp.shape[0]
    def loss(p):
        K = tsf_kernel(p, n)
        return np.linalg.norm(C_emp - K, ord='fro')**2
    x0 = np.array([0.5, 0.5, 1.0, 0.2])
    bounds = [(0,None),(0,None),(0,None),(0,None)]
    res = minimize(loss, x0, bounds=bounds, method='L-BFGS-B')
    return res.x, tsf_kernel(res.x, n), res.fun

lam_hat = estimate_lambda(scales)
params, C_fit, loss_val = fit_tsf(C_emp)
c0, beta, gamma, aord = params
print(f"Estimated lambda: {lam_hat:.4f}")
print("Fitted params:", {"c0":c0, "beta":beta, "gamma":gamma, "alpha_ord":aord})
print(f"Fit loss (Fro norm^2): {loss_val:.6f}")

# TSF score: correlation between vectorized C_emp and C_fit (excluding diagonal)
mask = ~np.eye(C_emp.shape[0], dtype=bool)
v_emp = C_emp[mask]; v_fit = C_fit[mask]
tsf_score = np.corrcoef(v_emp, v_fit)[0,1]
print(f"TSF score (corr off-diag): {tsf_score:.3f}")

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(C_emp, origin='lower', aspect='auto')
plt.title('Empirical C (zero-diagonal)'); plt.xlabel('j'); plt.ylabel('i')
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(C_fit, origin='lower', aspect='auto')
plt.title('Fitted TSF Kernel'); plt.xlabel('j'); plt.ylabel('i')
plt.colorbar()
plt.tight_layout(); plt.savefig(OUT/'empirical_vs_tsf_fit.png', dpi=160); plt.show()

# Save numeric outputs
np.save(OUT/'C_emp.npy', C_emp)
np.save(OUT/'C_fit.npy', C_fit)
with open(OUT/'tsf_params.json','w') as f:
    import json
    json.dump({"lambda":lam_hat, "c0":float(c0), "beta":float(beta), "gamma":float(gamma), "alpha_ord":float(aord), "loss":float(loss_val), "tsf_score":float(tsf_score)}, f, indent=2)
print("Saved: C_emp.npy, C_fit.npy, tsf_params.json and plots in", OUT.as_posix())

## Optional: Pull OpenNeuro datasets (e.g., LSD fMRI or EEG)
To download OpenNeuro datasets in Colab, uncomment and run the following cell. You can then adapt the loader
to extract time series (for fMRI via nilearn; for EEG via MNE-BIDS). **This can be large**; ensure runtime has space.

In [ ]:
## Uncomment to use OpenNeuro (e.g., LSD fMRI ds003059 or psilocybin ds006072)
# !pip install openneuro-py nilearn mne-bids
# import openneuro as on
# target_dir = 'data/openneuro'
# os.makedirs(target_dir, exist_ok=True)
# # Example: download a small subset or entire dataset (beware size)
# on.download(dataset='ds003059', target_dir=target_dir)  # LSD fMRI (Carhart-Harris et al.)
# # For EEG BIDS datasets, you can then parse with MNE-BIDS, e.g.:
# # from mne_bids import BIDSPath, read_raw_bids
# # bp = BIDSPath(root=target_dir, subject='01', task='rest', datatype='eeg')
# # raw_bids = read_raw_bids(bp, verbose=True)
# # ...then proceed with wavelet & TSF steps as above.

## Finished
You should now have TSF plots and parameters in the `tsf_outputs/` folder. Use them in your deck/paper.